In [8]:
model = "llama3.2:1b"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [9]:
from langchain_ollama import ChatOllama

# ADD HERE YOUR CODE
model = ChatOllama(model=model)

In [10]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(model="bge-m3")

In [11]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_or_create_collection("ai_model_book")

# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(
    client=client,
    collection_name="ai_model_book", 
    embedding_function=embedding_model
)

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)


# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = {"docs": format_docs} | prompt | model | StrOutputParser()

In [14]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(search_query, k=3)

print(docs)

[Document(id='76a880fd-14e9-4cdf-8166-ea83c8bcca9c', metadata={'moddate': '2019-05-07T15:51:31+00:00', 'page': 33, 'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'trapped': '/False', 'source': './AI_Book.pdf', 'creationdate': '2019-05-07T15:51:31+00:00', 'total_pages': 510, 'page_label': '8', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow'}, page_content='Types of Machine Learning Systems\nThere are so many different types of Machine Learning systems that it is useful to\nclassify them in broad categories based on:\n Whether or not they are trained with human supervision (supervised, unsuper\nvised, semisupervised, and Reinforcement Learning)\n Whether or not they can learn incrementally on the fly (online versus batch\nlearning)\n Whether they work by simply comparing new data points to known data points,\nor instead detect patterns in the training data and 

In [15]:
chain.invoke(docs)

'Based on the retrieved docs, the main themes in the "Types of Machine Learning Systems" section are:\n\n1. **Classification of Machine Learning Systems**: The docs discuss how Machine Learning systems can be classified into four major categories:\n\t* Supervised Learning\n\t* Unsupervised Learning\n\t* Semisupervised Learning\n\t* Reinforcement Learning\n2. **Key Characteristics of Each Category**: Each category has its unique characteristics, such as:\n\t* Supervised Learning: Includes labeled training data with desired solutions (labels), and aims to improve the model through prediction and generalization.\n\t* Unsupervised Learning: Does not rely on labeled training data, and focuses on discovering patterns in the data.\n3. **Machine Learning Process**: The docs explain the machine learning process, which involves:\n\t* Gathering data in a training set\n\t* Feeding the training set to a learning algorithm\n\t* Tuning parameters to fit the model to the training set\n\t* Generalizing

In [16]:
# Simple stream the chain output
for chunk in chain.stream(docs):
    print(chunk, end="", flush=True)

The main themes in the retrieved documentation on "Types of Machine Learning Systems" can be summarized as follows:

1. **Classification of Machine Learning Systems**: The document categorizes Machine Learning systems based on four major criteria: supervised, unsupervised, semisupervised, and Reinforcement Learning. This classification is not exclusive and can be combined in various ways.

2. **Understanding Supervised and Unsupervised Learning**: The document explains the difference between supervised and unsupervised learning, highlighting the role of labels and data in training Machine Learning algorithms. Supervised learning involves feeding the algorithm desired solutions (labels) and using those to train the model, while unsupervised learning involves identifying patterns in the training data.

3. **Machine Learning Fundamentals**: The document provides an overview of machine learning, emphasizing that it is about making machines better at tasks by learning from data. It explains

In [ ]:
# More complex async event streaming
async for event in chain.astream_events(docs, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

TypeError: 'ellipsis' object is not iterable

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [11]:
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

# ADD HERE YOUR CODE
rag_prompt = ...

# ADD HERE YOUR CODE
retriever = ...

# ADD HERE YOUR CODE
qa_rag_chain = ...

In [ ]:
qa_rag_chain

In [ ]:
question = "What is supervised learning?"

# ADD HERE YOUR CODE
qa_rag_chain.invoke(...)

In [ ]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [ ]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

In [ ]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [ ]:
qa_chain = ConversationalRetrievalChain.from_llm(
    model, retriever=retriever, memory=memory, verbose=False
)

In [ ]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

In [ ]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)